# Small MLP PYNQ 验证 — PicoRV32 交叉验证本 notebook 在 **PYNQ-Z2** 上运行，使用 PS + CIM 硬件对 small MLP (784→16→10)进行推理，作为 PicoRV32 仿真结果的 **交叉验证**。两条验证路径使用完全相同的：- 模型权重 / 偏置 / 量化参数- 测试图片- CIM 硬件 IP (cim_accel_core)预期结果：PS 推理与 PicoRV32 仿真 **bit-exact 一致**。## 前置条件1. `cim_soc.bit` + `cim_soc.hwh` 在当前目录2. `small_mlp_data/` 目录（从宿主机上传）3. `cim_driver.py`（从 sw/ 目录上传）4. `golden_results.txt`（从宿主机生成）

## 1. 加载 Overlay

In [ ]:
from pynq import Overlay, MMIO
import numpy as np
import time

ol = Overlay("cim_soc.bit")
print("Overlay loaded!")

BASE = 0x40000000
mmio = MMIO(BASE, 0x4000)

# 快速连通性测试
mmio.write(0x010, 784)
assert mmio.read(0x010) == 784, "AXI 读写失败!"
print("AXI connectivity OK")

## 2. CIM 驱动函数

In [ ]:
# ---- 硬件常量 ----
TILE_ROWS = 16
TILE_COLS = 16
ELEMS_PER_CHUNK = 4
CHUNKS_PER_ROW  = 4
CHUNKS_PER_TILE = 64

# ---- CSR 地址 ----
CTRL          = 0x000
STATUS        = 0x004
CSR_IN_DIM    = 0x010
CSR_OUT_DIM   = 0x014
CSR_N_IB      = 0x018
CSR_N_OB      = 0x01C
REQUANT_MULT  = 0x020
REQUANT_SHIFT = 0x024
INPUT_ZP      = 0x028
ACT_MODE      = 0x02C
PRED_CLASS    = 0x040
WDMA_ADDR     = 0x044
WDMA_DATA     = 0x048
WDMA_CTRL     = 0x04C
LOGIT_BASE    = 0x100
MEM_INPUT     = 0x1000
MEM_BIAS      = 0x2000

def soft_reset():
    mmio.write(CTRL, 0x4)

def configure(in_dim, out_dim, zp, mult, shift, relu):
    n_ib = (in_dim + 15) // 16
    n_ob = (out_dim + 15) // 16
    mmio.write(CSR_IN_DIM, in_dim)
    mmio.write(CSR_OUT_DIM, out_dim)
    mmio.write(CSR_N_IB, n_ib)
    mmio.write(CSR_N_OB, n_ob)
    mmio.write(REQUANT_MULT, mult)
    mmio.write(REQUANT_SHIFT, shift)
    mmio.write(INPUT_ZP, int(zp) & 0xFFFFFFFF)
    mmio.write(ACT_MODE, 1 if relu else 0)

def load_weights(chunks):
    mmio.write(WDMA_ADDR, 0)
    mmio.write(WDMA_CTRL, 0x02)
    for c in chunks:
        mmio.write(WDMA_DATA, int(c) & 0xFFFFFFFF)
    mmio.write(WDMA_CTRL, 0x00)

def load_bias(bias_u32, n):
    for i in range(n):
        mmio.write(MEM_BIAS + 4 * i, int(bias_u32[i]) & 0xFFFFFFFF)

def load_input(data_u8):
    padded_n = ((len(data_u8) + 15) // 16) * 16
    for i in range(padded_n):
        val = int(data_u8[i]) if i < len(data_u8) else 0
        mmio.write(MEM_INPUT + 4 * i, val)

def start_and_wait(timeout=5.0):
    mmio.write(CTRL, 0x2)  # clear done
    mmio.write(CTRL, 0x1)  # start
    t0 = time.time()
    while not (mmio.read(STATUS) & 0x2):
        if time.time() - t0 > timeout:
            raise TimeoutError("CIM 计算超时!")
    return time.time() - t0

def read_logits(out_dim):
    return [np.int8(mmio.read(LOGIT_BASE + 4*i) & 0xFF) for i in range(out_dim)]

def read_pred():
    return mmio.read(PRED_CLASS)

print("驱动函数定义完成")

## 3. 加载模型数据

In [ ]:
DATA_DIR = "small_mlp_data"

def read_hex_u32(path):
    with open(path) as f:
        return [int(l.strip(), 16) for l in f if l.strip()]

def read_hex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

# 量化参数
qp = read_hex_u32(f"{DATA_DIR}/quant_params.hex")
zps = read_hex_u32(f"{DATA_DIR}/zero_points.hex")
fc1_mult, fc1_shift = qp[0], qp[1]
fc2_mult, fc2_shift = qp[2], qp[3]
hw_zp1 = np.int32(np.uint32(zps[0]))
hw_zp2 = np.int32(np.uint32(zps[1]))

# 权重 / 偏置
fc1_wc = read_hex_u32(f"{DATA_DIR}/fc1_weight_tiles.hex")
fc2_wc = read_hex_u32(f"{DATA_DIR}/fc2_weight_tiles.hex")
fc1_bias = read_hex_u32(f"{DATA_DIR}/fc1_bias.hex")
fc2_bias = read_hex_u32(f"{DATA_DIR}/fc2_bias.hex")

print(f"FC1: {len(fc1_wc)} weight chunks, {len(fc1_bias)} bias, mult={fc1_mult}, shift={fc1_shift}")
print(f"FC2: {len(fc2_wc)} weight chunks, {len(fc2_bias)} bias, mult={fc2_mult}, shift={fc2_shift}")
print(f"ZP: fc1={hw_zp1}, fc2={hw_zp2}")

## 4. 推理 20 张测试图

In [ ]:
import os

# 检测有多少测试图
test_dir = f"{DATA_DIR}/test_images"
n_images = len([f for f in os.listdir(test_dir) if f.endswith("_label.txt")])
print(f"找到 {n_images} 张测试图")

results = []

for i in range(n_images):
    prefix = f"img_{i:04d}"
    img = np.array(read_hex_u8(f"{test_dir}/{prefix}.hex"), dtype=np.uint8)
    label = int(open(f"{test_dir}/{prefix}_label.txt").read().strip())

    # ---- FC1: 784→16, ReLU ----
    soft_reset()
    configure(784, 16, hw_zp1, fc1_mult, fc1_shift, relu=True)
    load_weights(fc1_wc)
    load_bias(fc1_bias, 16)
    load_input(img)
    t1 = start_and_wait()

    fc1_out = np.array(read_logits(16), dtype=np.int8)
    fc1_out_u8 = fc1_out.view(np.uint8)

    # ---- FC2: 16→10, no ReLU ----
    configure(16, 10, hw_zp2, fc2_mult, fc2_shift, relu=False)
    load_weights(fc2_wc)
    load_bias(fc2_bias, 10)
    load_input(fc1_out_u8)
    t2 = start_and_wait()

    logits = read_logits(10)
    pred = int(np.argmax(logits))

    ok = "✓" if pred == label else "✗"
    results.append({
        "idx": i, "label": label, "pred": pred,
        "correct": pred == label,
        "logits": list(logits),
        "time_ms": (t1 + t2) * 1000
    })
    print(f"  [{i:04d}] label={label}, pred={pred} {ok}  logits={list(logits)}  ({(t1+t2)*1000:.1f}ms)")

correct = sum(1 for r in results if r["correct"])
print(f"\nPYNQ 硬件准确率: {correct}/{n_images} ({100*correct/n_images:.0f}%)")

## 5. 与 PicoRV32 仿真结果对比

In [ ]:
# 读取宿主机 golden model 结果 (如果有)
golden_file = "golden_results.txt"
golden = {}
if os.path.exists(golden_file):
    with open(golden_file) as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split()
            if len(parts) == 4:
                golden[int(parts[0])] = {
                    "label": int(parts[1]),
                    "pred":  int(parts[2]),
                    "correct": int(parts[3]) == 1
                }
    print(f"已加载 golden results ({len(golden)} 条)")
else:
    print(f"未找到 {golden_file}, 跳过对比")

# 对比
print()
print(f"{'Image':>6s} {'Label':>6s} {'PYNQ':>6s} {'Golden':>7s} {'Match':>6s}")
print("-" * 40)
all_match = True
for r in results:
    i = r["idx"]
    pynq_pred = r["pred"]
    if i in golden:
        g_pred = golden[i]["pred"]
        match = "✓" if pynq_pred == g_pred else "✗ MISMATCH"
        if pynq_pred != g_pred:
            all_match = False
    else:
        g_pred = "?"
        match = "-"
    print(f"{i:6d} {r['label']:6d} {pynq_pred:6d} {str(g_pred):>7s} {match:>6s}")

print("-" * 40)
hw_correct = sum(1 for r in results if r["correct"])
print(f"PYNQ 准确率: {hw_correct}/{len(results)}")

if golden:
    g_correct = sum(1 for v in golden.values() if v["correct"])
    print(f"Golden 准确率: {g_correct}/{len(golden)}")
    if all_match:
        print("\n★ PYNQ 与 Golden Model 结果完全一致 (bit-exact) ★")
    else:
        print("\n⚠ 存在不一致! 请检查硬件配置")

## 结论如果所有结果 bit-exact 一致，说明:1. **CIM 硬件 IP** 在 PYNQ 板上工作正确2. **PicoRV32 仿真** 与 **PS 驱动** 使用的是完全相同的硬件逻辑3. Small MLP (784→16→10) 端到端推理验证通过这证明了 PicoRV32 RISC-V 软核可以替代 ARM PS 完成 CIM 加速器的控制。